In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import wandb

import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

## Load ground truth data

In [ ]:
BD = ButterflyDataset(
    modalities={"coords": None}, data_dir=os.path.join(os.environ["DATA_DIR"]), use_aux_data="all"
)

BCB = ButterflyCaptionBuilder(
    data_dir=os.path.join(os.environ["DATA_DIR"], "s2bms"),
    templates_fname="v5.json",
    concepts_fname="v6.json",
    seed=42,
)

BM = BaseDataModule(
    dataset=BD,
    batch_size=32,
    split_mode="from_file",
    saved_split_file_name=os.path.join(
        os.environ["DATA_DIR"], "s2bms", "splits/split_indices_s2bms_2024-08-14-1459.pth"
    ),
    caption_builder=BCB,
)

split = BM.load_split_indices(
    os.path.join(os.environ["DATA_DIR"], "s2bms", "splits/split_indices_s2bms_2024-08-14-1459.pth")
)
print(split.keys())

In [ ]:
## get a df with the test data:

df_test = BD.df[np.isin(BD.df.name_loc, split["test_indices"].values)]
df_test_aux = df_test.copy()
df_test.drop([c for c in df_test.columns if "aux_" in c], axis=1, inplace=True)
cols_targets = [c for c in df_test.columns if "target_" in c]
df_test_aux.rename(columns={c: f"target_{ii}" for ii, c in enumerate(cols_targets)}, inplace=True)
df_test.rename(columns={c: f"target_{ii}" for ii, c in enumerate(cols_targets)}, inplace=True)

## Create context for inference captions:

In [ ]:
corine_context = pd.read_csv(os.path.join(os.environ["DATA_DIR"], "s2bms", "corine_classes.csv"))
context_dict = {}
for i, row in corine_context.iterrows():
    code = row["code"]
    for ii in range(3):
        new_code = code[: (len(code) - ii)]
        if new_code not in context_dict:
            context_dict[new_code[4:]] = row["category_level_{}".format(3 - ii)]

bioclim_context = pd.read_csv(os.path.join(os.environ["DATA_DIR"], "s2bms", "bioclim_classes.csv"))

for i, row in bioclim_context.iterrows():
    code = row["name"][4:]
    context_dict[code] = row["description"].split(" (")[0]

context_dict = {
    **context_dict,
    **{
        "maxdist_road": "Maximum distance to road",
        "meandist_road": "Mean distance to road",
        "p_arable": "Arable land",
        "p_broadleaf": "Broadleaf forest",
        "p_coast": "Coastline",
        "p_conifer": "Coniferous forest",
        "p_mountain": "Mountainous terrain",
        "p_smnat_grsl": "Semi-natural grassland",
        "p_urban": "Urban areas",
        "pm252024g": "PM2.5 pollution",
        "pop_density": "Population density",
        "shdi_lc": "Shannon diversity index (land cover)",
        "total_population": "Total population",
    },
}

In [ ]:
model_config = pd.read_csv(
    os.environ["PROJECT_ROOT"] + "/inference/s2bms/v1/" + "best_aef_run_config_v10.csv"
)
model_config.drop(
    [c for c in model_config.columns if model_config[c].dtype == "object" or "k_index" not in c],
    axis=1,
    inplace=True,
)
model_config.drop([c for c in model_config.columns if "avr" in c], axis=1, inplace=True)
model_config.drop(
    [
        c
        for c in model_config
        if any(
            [
                s in c
                for s in [
                    "test_dyn_k_index_elevation_max",
                    "test_dyn_k_index_river_dens_max",
                    "test_dyn_k_index_slope_max",
                    "test_dyn_k_index_dist_to_sea_max",
                ]
            ]
        )
    ],
    axis=1,
    inplace=True,
)
model_config.drop(
    [c for c in model_config if pd.isna(model_config[c]).any()], axis=1, inplace=True
)
# get mean sem for each column
model_config_mean_sem = pd.DataFrame()
for col in model_config.columns:
    model_config_mean_sem.loc[0, col] = model_config[col].mean()
    model_config_mean_sem.loc[1, col] = model_config[col].sem()


dict_steps = {
    "Use preferably:": (0.67, 1),
    "Use if needed:": (0.33, 0.67),
    "Do not use:": (-10, 0.33),
}

for step, (low, high) in dict_steps.items():
    cols_selection = [
        c for c in model_config_mean_sem.columns if low < model_config_mean_sem.loc[0, c] <= high
    ]
    cols_selection = sorted(
        cols_selection, key=lambda x: model_config_mean_sem.loc[0, x], reverse=True
    )

    print(f"\n\n{step}\n")
    for c in cols_selection:
        print(
            f"{context_dict[c[len('test_dyn_k_index_'):-4]]}: {model_config_mean_sem.loc[0, c]:.3f}"
        )

## Load habitat inference captions

In [ ]:
# all_captions = [h['inference_captions'] for h in habitat_preferences]
# all_captions = [item for sublist in all_captions for item in sublist]
# titles = [[h['common_name']] * len(h['inference_captions']) for h in habitat_preferences]
# titles = [item for sublist in titles for item in sublist]
# assert len(all_captions) == len(set(all_captions))

## Load inference results:

In [ ]:
list_hab_pref_files = [
    "habitat_preferences_v3",
    "habitat_preferences_v4",
    "habitat_preferences_v5",
    "habitat_preferences_v6",
    "habitat_preferences_v7",
    "habitat_preferences_v8",
]

dict_results_inf_save = {
    x: [] for x in ["version", "top_5_acc_pred", "top_5_acc_sim_mean", "top_5_acc_sim_max"]
}

for hab_pref_file in list_hab_pref_files:
    dict_results_inf_save["version"].append(hab_pref_file)
    with open(
        os.path.join(
            os.environ["PROJECT_ROOT"], "data/s2bms", f"inference_captions/{hab_pref_file}.json"
        ),
        "r",
    ) as f:
        habitat_preferences = json.load(f)

    name_to_id = {h["common_name"]: h["target_id"] for h in habitat_preferences}

    folder_inference_results = os.path.join(os.environ["PROJECT_ROOT"], "inference/s2bms/v1/")
    tmp_files = [
        f
        for f in os.listdir(folder_inference_results)
        if f.endswith(".csv") and hab_pref_file in f
    ]
    assert (
        len(tmp_files) == 1
    ), f"Expected one inference results file for {hab_pref_file}, found {len(tmp_files)} in {folder_inference_results}: {tmp_files}"
    fp_inference_results = os.path.join(folder_inference_results, tmp_files[0])
    assert os.path.exists(fp_inference_results), f"File not found: {fp_inference_results}"

    df_inference = pd.read_csv(fp_inference_results)
    df_inference.drop("Unnamed: 0", axis=1, inplace=True)
    df_inference_test = df_inference[np.isin(df_inference.name_loc, split["test_indices"].values)]

    species = set(
        [
            c.split("_")[0]
            for c in df_inference_test.columns
            if not c.startswith("target_") and not c.startswith("name_loc")
        ]
    )
    species = list(species)
    targets_keep = [f"target_{name_to_id[s]}" for s in species]

    for s in species:
        cols_sim = [c for c in df_inference_test.columns if c.startswith(f"{s}_")]
        ## col average:
        df_inference_test[f"sim_max_{s}"] = df_inference_test[cols_sim].max(1)
        df_inference_test[f"sim_mean_{s}"] = df_inference_test[cols_sim].mean(1)
        df_inference_test.drop(cols_sim, axis=1, inplace=True)

        id_species = name_to_id[s]
        val_species = df_test[f"target_{id_species}"].values
        df_inference_test[f"true_{s}"] = val_species

    df_inference_test.drop(
        [
            c
            for c in df_inference_test.columns
            if c.startswith("target_") and c not in targets_keep
        ],
        axis=1,
        inplace=True,
    )
    df_inference_test.rename(
        columns={f"target_{name_to_id[s]}": f"pred_{s}" for s in species}, inplace=True
    )

    if hab_pref_file == "habitat_preferences_v6":
        df_inference_test_v6 = df_inference_test.copy()

    list_top_5_acc_pred = []
    list_top_5_acc_sim_max = []
    list_top_5_acc_sim_mean = []
    # list_top_5_agreement_pred_sim = []
    list_top_5_acc_meanrate = []
    # list_top_k_acc_predsim = []
    # list_top_k_acc_pred = []
    mat_top_5_species_prob = np.zeros((len(df_inference_test), 5))

    top_5_species_overall = sorted(
        species, key=lambda s: df_inference_test[f"pred_{s}"].sum(), reverse=True
    )[:5]

    for i, row in enumerate(df_inference_test.iterrows()):
        top_5_species = sorted(species, key=lambda s: row[1][f"true_{s}"], reverse=True)[:5]
        pred_top_5_species = sorted(species, key=lambda s: row[1][f"pred_{s}"], reverse=True)[:5]
        probs_top_5_species = np.array([row[1][f"pred_{s}"] for s in pred_top_5_species])
        mat_top_5_species_prob[i, :] = probs_top_5_species
        sim_top_5_species_max = sorted(
            species, key=lambda s: row[1][f"sim_max_{s}"], reverse=True
        )[:5]
        sim_top_5_species_mean = sorted(
            species, key=lambda s: row[1][f"sim_mean_{s}"], reverse=True
        )[:5]

        # predsim_top_k = list(set(pred_top_5_species) | set(sim_top_5_species))
        # top_k_species = sorted(species, key=lambda s: row[1][f'true_{s}'], reverse=True)[:len(predsim_top_k)]
        # pred_top_k_species = sorted(species, key=lambda s: row[1][f'pred_{s}'], reverse=True)[:len(predsim_top_k)]

        top_5_acc_pred = len(set(top_5_species) & set(pred_top_5_species)) / 5
        top_5_acc_sim_max = len(set(top_5_species) & set(sim_top_5_species_max)) / 5
        top_5_acc_sim_mean = len(set(top_5_species) & set(sim_top_5_species_mean)) / 5
        top_5_acc_baseline = len(set(top_5_species_overall) & set(top_5_species)) / 5
        # top_5_agreement_pred_sim = len(set(pred_top_5_species) & set(sim_top_5_species)) / 5
        # top_k_acc_predsim = len(set(top_k_species) & set(predsim_top_k)) / len(top_k_species) if len(top_k_species) > 0 else np.nan

        list_top_5_acc_pred.append(top_5_acc_pred)
        list_top_5_acc_sim_max.append(top_5_acc_sim_max)
        list_top_5_acc_sim_mean.append(top_5_acc_sim_mean)
        list_top_5_acc_meanrate.append(top_5_acc_baseline)
        # list_top_5_agreement_pred_sim.append(top_5_agreement_pred_sim)
        # list_top_k_acc_predsim.append(top_k_acc_predsim)

    dict_results_inf_save["top_5_acc_pred"].append(np.mean(list_top_5_acc_pred))
    dict_results_inf_save["top_5_acc_sim_max"].append(np.mean(list_top_5_acc_sim_max))
    dict_results_inf_save["top_5_acc_sim_mean"].append(np.mean(list_top_5_acc_sim_mean))
    # print(f"\n\nResults for {hab_pref_file}:\n")
    # print(f"Top 5 accuracy (pred): {np.mean(list_top_5_acc_pred):.3f}")
    # print(f"Top 5 accuracy (sim): {np.mean(list_top_5_acc_sim):.3f}")
    # print(f"Top 5 accuracy (mean rate): {np.mean(list_top_5_acc_meanrate):.3f}")
    # print(f"Top 5 agreement (pred vs sim): {np.mean(list_top_5_agreement_pred_sim):.3f}")
    # print(f"Top k accuracy (pred vs sim): {np.nanmean(list_top_k_acc_predsim):.3f}")


map_versions = {
    "habitat_preferences_v3": "complex, 5 max",
    "habitat_preferences_v4": "simple, 5 max",
    "habitat_preferences_v5": "complex, 1 max",
    "habitat_preferences_v6": "simple, 10 max",
    "habitat_preferences_v7": "simple, 1 max",
    "habitat_preferences_v8": "complex, 10 max",
}

df_results_inf_save = pd.DataFrame(dict_results_inf_save)
df_results_inf_save["version"] = df_results_inf_save["version"].map(map_versions)
df_results_inf_save = df_results_inf_save.sort_values(by="top_5_acc_sim_max")
df_results_inf_save = df_results_inf_save.set_index("version")

## make new table with pred, simple 5 mean, complex 5 mean, simple 5 max, complex 5 max
## (2 columns: name + score)

n_captions = 10

df_results_simplified = pd.DataFrame(
    {
        "name": [
            "AEF prediction",
            f"Mean of {n_captions} simple captions",
            f"Mean of {n_captions} complex captions",
            f"Max of {n_captions} simple captions",
            f"Max of {n_captions} complex captions",
        ],
        "score": [
            np.mean(list_top_5_acc_pred),
            df_results_inf_save.loc[f"simple, {n_captions} max", "top_5_acc_sim_mean"],
            df_results_inf_save.loc[f"complex, {n_captions} max", "top_5_acc_sim_mean"],
            df_results_inf_save.loc[f"simple, {n_captions} max", "top_5_acc_sim_max"],
            df_results_inf_save.loc[f"complex, {n_captions} max", "top_5_acc_sim_max"],
        ],
    }
)

df_results_simplified = df_results_simplified.sort_values(by="score", ascending=True).reset_index(
    drop=True
)
df_results_simplified

In [ ]:
df_inference_test

In [ ]:
hab_pref_file = "habitat_preferences_v6"

dict_results_inf_save["version"].append(hab_pref_file)
with open(
    os.path.join(
        os.environ["PROJECT_ROOT"], "data/s2bms", f"inference_captions/{hab_pref_file}.json"
    ),
    "r",
) as f:
    habitat_preferences = json.load(f)

name_to_id = {h["common_name"]: h["target_id"] for h in habitat_preferences}

folder_inference_results = os.path.join(os.environ["PROJECT_ROOT"], "inference/s2bms/v1/")
tmp_files = [
    f for f in os.listdir(folder_inference_results) if f.endswith(".csv") and hab_pref_file in f
]
assert (
    len(tmp_files) == 1
), f"Expected one inference results file for {hab_pref_file}, found {len(tmp_files)} in {folder_inference_results}: {tmp_files}"
fp_inference_results = os.path.join(folder_inference_results, tmp_files[0])
assert os.path.exists(fp_inference_results), f"File not found: {fp_inference_results}"

df_inference = pd.read_csv(fp_inference_results)
df_inference.drop("Unnamed: 0", axis=1, inplace=True)
df_inference_test = df_inference[np.isin(df_inference.name_loc, split["test_indices"].values)]

species = set(
    [
        c.split("_")[0]
        for c in df_inference_test.columns
        if not c.startswith("target_") and not c.startswith("name_loc")
    ]
)
species = list(species)
targets_keep = [f"target_{name_to_id[s]}" for s in species]

for s in species:
    cols_sim = [c for c in df_inference_test.columns if c.startswith(f"{s}_")]
    ## col average:
    df_inference_test[f"sim_max_{s}"] = df_inference_test[cols_sim].max(1)
    df_inference_test[f"sim_mean_{s}"] = df_inference_test[cols_sim].mean(1)
    # df_inference_test.drop(cols_sim, axis=1, inplace=True)

    id_species = name_to_id[s]
    val_species = df_test[f"target_{id_species}"].values
    df_inference_test[f"true_{s}"] = val_species

df_inference_test.drop(
    [c for c in df_inference_test.columns if c.startswith("target_") and c not in targets_keep],
    axis=1,
    inplace=True,
)
df_inference_test.rename(
    columns={f"target_{name_to_id[s]}": f"pred_{s}" for s in species}, inplace=True
)

df_inference_test_v6 = df_inference_test.copy()
cols_v6 = df_inference_test_v6.columns
df_inference_test_v6_sel = df_inference_test_v6[[c for c in cols_v6 if "Dingy" in c]]

sns.scatterplot(data=df_inference_test_v6_sel, y="pred_Dingy Skipper", x="true_Dingy Skipper")
caption_argmax = df_inference_test_v6_sel.values[:, :10].argmax(1)
hab_captions = habitat_preferences[1]["inference_captions"]

from collections import Counter

counter_argmax = Counter(caption_argmax)
for i, c in enumerate(hab_captions):
    print(f"Caption {i}: {c} - Count: {counter_argmax[i]}")

In [ ]:
# for s in species:
#     fig, ax = plt.subplots(figsize=(6, 6))
#     try:
#         sns.scatterplot(data=df_inference_test_v6, y=f'pred_{s}', x=f'true_{s}', ax=ax)
#     except Exception as e:
#         print(f"Error plotting for species {s}: {e}")
#         continue

In [ ]:
print(f"aux_vals shape: {concept_vals.shape}, target_vals shape: {target_vals.shape}")
concept_by_target_corr = np.corrcoef(concept_vals.T, target_vals.T)[
    : concept_vals.shape[1], concept_vals.shape[1] :
]
print(f"concept_by_target_corr shape: {concept_by_target_corr.shape}")

In [ ]:
aux_cols = [c for c in df_test_aux.columns if "aux_" in c and "top" not in c]
target_cols = [c for c in df_test_aux.columns if "target_" in c]


df_inference_results_concepts = pd.read_csv(
    os.path.join(folder_inference_results, "2026-09-25_10-23-35_concept_captions_v10.csv")
)

tmp_save = []
for con in aux_cols:
    cols_sim = [c for c in df_inference_results_concepts.columns if con in c]
    ## col average:
    df_inference_results_concepts[f"sim_max_{con}"] = df_inference_results_concepts[cols_sim].max(
        1
    )
    df_inference_results_concepts[f"sim_mean_{con}"] = df_inference_results_concepts[
        cols_sim
    ].mean(1)
    df_inference_results_concepts.drop(cols_sim, axis=1, inplace=True)
    tmp_save.append(f"sim_max_{con}")
    # tmp_save.append(f'sim_mean_{con}')

df_inference_results_concepts = df_inference_results_concepts[
    np.isin(df_inference_results_concepts.name_loc, split["test_indices"].values)
]
# df_inference_results_concepts = df_inference_results_concepts[['name_loc'] + tmp_save]
df_inference_results_concepts = df_inference_results_concepts[tmp_save]
df_inference_results_concepts.dropna(axis=1, inplace=True)
# df_inference_results_concepts


concept_vals = df_inference_results_concepts.values
target_vals = df_test_aux[target_cols].values

print(f"aux_vals shape: {concept_vals.shape}, target_vals shape: {target_vals.shape}")
print(f"aux_vals shape: {concept_vals.shape}, target_vals shape: {target_vals.shape}")
concept_ranks = pd.DataFrame(concept_vals).rank(axis=0).values
target_ranks = pd.DataFrame(target_vals).rank(axis=0).values
concept_by_target_corr = np.corrcoef(concept_ranks.T, target_ranks.T)[
    : concept_ranks.shape[1], concept_ranks.shape[1] :
]
# concept_by_target_corr = np.corrcoef(concept_vals.T, target_vals.T)[:concept_vals.shape[1], concept_vals.shape[1]:]

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
sns.heatmap(
    concept_by_target_corr,
    xticklabels=target_cols,
    yticklabels=df_inference_results_concepts.columns,
    cmap="coolwarm",
    center=0,
    ax=ax[0],
)


aux_cols = [c for c in aux_cols if f"sim_max_{c}" in df_inference_results_concepts.columns]
aux_vals = df_test_aux[aux_cols].values
target_vals = df_test_aux[target_cols].values

print(f"aux_vals shape: {aux_vals.shape}, target_vals shape: {target_vals.shape}")
aux_ranks = pd.DataFrame(aux_vals).rank(axis=0).values
target_ranks = pd.DataFrame(target_vals).rank(axis=0).values
aux_by_target_corr = np.corrcoef(aux_ranks.T, target_ranks.T)[
    : aux_ranks.shape[1], aux_ranks.shape[1] :
]
# aux_by_target_corr = np.corrcoef(aux_vals.T, target_vals.T)[:len(aux_cols), len(aux_cols):]

sns.heatmap(
    aux_by_target_corr,
    xticklabels=target_cols,
    yticklabels=aux_cols,
    cmap="coolwarm",
    center=0,
    ax=ax[1],
)

ax[0].set_title("Concepts by target correlation")
ax[1].set_title("Auxiliary data by target correlation")

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.plot(aux_by_target_corr.flatten(), concept_by_target_corr.flatten(), "o", alpha=0.5)
ax.set_xlabel("Species/auxiliary data correlation")
ax.set_ylabel("Species/concept retrieval correlation")

In [ ]:
save_figure = False

list_corr_sim_pred = []
list_corr_true_sim = []
species_plot = []

for s in species:
    try:
        arr_true = df_inference_test_v6[f"true_{s}"].values
        arr_pred = df_inference_test_v6[f"pred_{s}"].values
        arr_sim = df_inference_test_v6[f"sim_max_{s}"].values
        species_plot.append(s)
    except Exception as e:
        print(f"Error processing species {s}: {e}")
        continue

    corr_sim_pred = np.corrcoef(arr_sim, arr_pred)[0, 1]
    corr_true_sim = np.corrcoef(arr_true, arr_sim)[0, 1]
    list_corr_sim_pred.append(corr_sim_pred)
    list_corr_true_sim.append(corr_true_sim)
    print(f"Correlation for {s}: {corr_sim_pred:.3f}, {corr_true_sim:.3f}")

# plt.plot(list_corr_sim_pred, label='sim vs pred')
# plt.plot(list_corr_true_sim, label='true vs sim')
# plt.xlabel('Species')
# plt.ylabel('Correlation')
# plt.legend()
# plt.show()


# plot a pair plot of correlation values:
corr_sim_labels = np.squeeze(list_corr_true_sim)
corr_sim_preds = np.squeeze(list_corr_sim_pred)
n = len(corr_sim_preds)


import scipy.stats as stats

inds_nonnan = ~np.isnan(corr_sim_labels) & ~np.isnan(corr_sim_preds)
ttest = stats.ttest_rel(
    corr_sim_labels[inds_nonnan], corr_sim_preds[inds_nonnan], alternative="less"
)
print(f"Paired t-test: t={ttest.statistic:.3f}, p={ttest.pvalue:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 2), gridspec_kw={"width_ratios": [1, 2], "wspace": 0.5})


aux_by_target_corr_plot = aux_by_target_corr.flatten()
concept_by_target_corr_plot = concept_by_target_corr.flatten()
inds_nonnan = ~np.isnan(aux_by_target_corr_plot) & ~np.isnan(concept_by_target_corr_plot)
aux_by_target_corr_plot = aux_by_target_corr_plot[inds_nonnan]
concept_by_target_corr_plot = concept_by_target_corr_plot[inds_nonnan]

r, p = stats.pearsonr(aux_by_target_corr_plot, concept_by_target_corr_plot)
print(f"Pearson correlation: r={r:.3f}, p={p}")

h = ax[0].hist2d(
    aux_by_target_corr_plot,
    concept_by_target_corr_plot,
    bins=20,
    cmap="Blues",
)
fig.colorbar(h[3], ax=ax[0], label="Count")
ax[0].set_xlabel(r"$\rho$" + "(species, auxiliary data)")
ax[0].set_ylabel(r"$\rho$" + "(species, aux. concepts)")

ax[1].bar(
    np.arange(n) - 0.2,
    corr_sim_labels,
    width=0.4,
    label="True occurrence rates",
    color="blue",
    alpha=0.5,
    edgecolor="k",
)
ax[1].bar(
    np.arange(n) + 0.2,
    corr_sim_preds,
    width=0.4,
    label="Predicted occurrence rates",
    color="blue",
    edgecolor="k",
)
# ax.set_xlabel('Samples')
ax[1].set_ylabel(r"$\rho$" + "(embedding similarity,\noccurrence rates)")
ax[1].legend()
ax[1].set_xticks(np.arange(n))
ax[1].set_xticklabels(species_plot, rotation=30, ha="right")

for sp in ["top", "right", "bottom"]:
    ax[1].spines[sp].set_visible(False)

import src.utils.viz_utils as vu

ax[0].set_title("Auxiliary data \nqueried with concepts", fontsize=10, fontdict={"weight": "bold"})
ax[1].set_title(
    "OVQ concepts explain predictions\nfor some species", fontsize=10, fontdict={"weight": "bold"}
)

vu.add_panel_label(ax[0], label_letter="a", fontsize=14, x_offset=0.38, y_offset=0.13)
vu.add_panel_label(ax[1], label_letter="b", fontsize=14, x_offset=0.99, y_offset=0.13)


if save_figure:
    fig.savefig(
        f"/Users/tplas/repos/ms_aether_biodiv/figs/concepts_aux_data_correlation.pdf",
        dpi=300,
        bbox_inches="tight",
    )

In [ ]:
fig, ax = plt.subplots(4, 4, figsize=(25, 25), gridspec_kw={"wspace": 0.2, "hspace": 0.2})
ax = ax.flatten()

for s in species:
    ax_i = species.index(s)
    sns.scatterplot(data=df_inference_test, x=f"pred_{s}", y=f"sim_{s}", ax=ax[ax_i])
    ax[ax_i].set_title(f"{s} (id: {name_to_id[s]})")
    ax[ax_i].set_xlabel("true value")
    ax[ax_i].set_ylabel("sim value")
    # ax[ax_i].plot([0, 1], [0, 1], 'r--')

In [ ]:
fig, ax = plt.subplots(8, 8, figsize=(25, 25), gridspec_kw={"wspace": 0.1, "hspace": 0.1})
ax = ax.flatten()

for i_target, target in enumerate(target_names):
    curr_ax = ax[i_target]
    curr_ax.plot(df_test[f"target_{i_target}"], df_inference_test[f"target_{i_target}"], ".")
    curr_ax.set_xlim([-0.05, 1.05])
    curr_ax.set_ylim([-0.05, 1.05])
    curr_ax.set_xlabel("")
    curr_ax.set_ylabel("")
    curr_ax.set_xticklabels([])
    curr_ax.set_yticklabels([])

In [ ]:
hab_pref_file = "habitat_preferences_v6"
with open(
    os.path.join(
        os.environ["PROJECT_ROOT"], "data/s2bms", f"inference_captions/{hab_pref_file}.json"
    ),
    "r",
) as f:
    habitat_preferences = json.load(f)

for h in habitat_preferences:
    print(len(h["inference_captions"]))

## transform concept captions:


In [ ]:
fp_concept = os.path.join(os.environ["PROJECT_ROOT"], "data/s2bms", "concept_captions/v10.json")
assert os.path.exists(fp_concept), f"Concept captions file not found: {fp_concept}"

with open(fp_concept, "r") as f:
    concept_captions = json.load(f)

concept_captions

inference_captions = []
for c in concept_captions:
    inference_captions.append(
        {"inference_captions": c["concept_caption"], "common_name": c["col"]}
    )


with open(
    os.path.join(
        os.environ["PROJECT_ROOT"], "data/s2bms", "inference_captions/concept_captions_v10.json"
    ),
    "w",
) as f:
    json.dump(inference_captions, f, indent=4)